# SBERT

En este cuadernillo se construye un sistema de recomendación basado en contenido, usando SBERT para generar embeddings de las películas a partir de su overview, tags y genres, y calculando después la similitud del coseno entre esos embeddings.

Documentación:
- https://sbert.net/
- https://sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html

In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\lucia\OneDrive\Escritorio\SR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carga de datos
Se carga el dataset de PLN (NLP.parquet) con la información de cada película, y los conjuntos de ratings. Train y val se unen en un único conjunto de entrenamiento, y test se mantiene aparte para la evaluación final.

In [ ]:
df = pd.read_parquet("../../data/03_model_ready/NLP.parquet").sort_values('movieId')
t =  pd.read_parquet("../../data/03_model_ready/ratings_train.parquet")
val =  pd.read_parquet("../../data/03_model_ready/ratings_val.parquet")
train = pd.concat([t, val], ignore_index=True)
test =  pd.read_parquet("../../data/03_model_ready/ratings_test.parquet")
df.head()

,movieId,title,genres,tag,overview,join
0,1,Toy Story,adventure animation children comedy fantasy,pixar pixar fun,"Led by Woody, Andy's toys live happily in his ...",adventure animation children comedy fantasy pi...
1,2,Jumanji,adventure children fantasy,fantasy magic board game robin williams game,When siblings Judy and Peter discover an encha...,adventure children fantasy fantasy magic board...
2,3,Grumpier Old Men,comedy romance,moldy old,A family wedding reignites the ancient feud be...,comedy romance moldy old A family wedding reig...
3,4,Waiting to Exhale,comedy drama romance,,"Cheated on, mistreated and stepped on, the wom...","comedy drama romance Cheated on, mistreated a..."
4,5,Father of the Bride Part II,comedy,pregnancy remake,Just when George Banks has recovered from his ...,comedy pregnancy remake Just when George Banks...


## Generación de embeddings
Se carga el modelo preentrenado all-MiniLM-L6-v2, que codifica cada texto en un vector de embeddings.

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2") #dim vectores   

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3637.49it/s]


Para cada película se construye un único texto uniendo overview, tag y genres, y se calcula su embedding con el modelo. El resultado es una matriz de embeddings con una fila por película.

In [ ]:
sentences = ( df['overview']  + " "+ df['tag'] + " " + df['genres']).tolist()
embeddings = model.encode(sentences,convert_to_tensor=True )

## Perfil de usuario y recomendación
El perfil de un usuario se define como la media de los embeddings de las películas que ha valorado en train, ponderada por su rating. Así, las películas mejor valoradas por el usuario tienen más peso en su perfil.

In [16]:
#vector de perfil de un usuario:
def basedUserProfile(userId):
    filtro = train[train['userId']==userId]
    pelisVistas = filtro['movieId'].tolist()
    pesos = filtro['rating'].tolist()
    indPelVistas = []
    for movieId in pelisVistas:
        indice = df[df['movieId']==movieId].index[0]
        indPelVistas.append(indice)
    vectores = embeddings[indPelVistas]
    return np.average(vectores, axis=0, weights=pesos)  

Para recomendar, se calcula la similitud del coseno entre el perfil del usuario y los embeddings de todas las películas. Las películas que el usuario ya ha visto se excluyen (se les asigna similitud -1), y se devuelven las 10 con mayor similitud.

In [17]:
#recomendacion para un usuario
def recomendacion(userId):
    UserProfile = basedUserProfile(userId)
    pelisVistas = train[train['userId']==userId]['movieId'].tolist()
    indPelVistas = df[df['movieId'].isin(pelisVistas)].index.tolist()
    similitud  = cosine_similarity(UserProfile.reshape(1, -1), embeddings).flatten()
    similitud[indPelVistas] = -1
    return np.argsort(similitud)[::-1][:10]

## Evaluación de las recomendaciones
Para cada usuario se generan las k=10 recomendaciones y se comparan con las películas de test que ha valorado. Se considera relevante una película de test si su rating es mayor o igual que el umbral t (por defecto 3.5). Con esto se calculan tres métricas por usuario:
- precisionK: proporción de las k recomendaciones que resultan relevantes.
- recalK: proporción de las películas relevantes de test que aparecen entre las recomendaciones.
- F1K: media armónica de precisionK y recalK.

resultadosUmbral calcula estas métricas para todos los usuarios y devuelve su media (multiplicada por p, útil para expresar el resultado en porcentaje).

In [7]:
def evaluacion(userId, k=10, t=3.5):
    indRec = recomendacion(userId)
    movieIdRec = df.iloc[indRec]['movieId'].values
    interseccion = test[(test['userId']==userId) & (test['movieId'].isin(movieIdRec))]
    
    nRelevantes = (interseccion['rating'] >= t).sum()
    nTest = (test[test["userId"]==userId]['rating'] >= t).sum() #pelis test relevantes
    precisionk = nRelevantes/k
    if nTest !=0:
        recalk = nRelevantes/nTest
    else:
        recalk = 0
    if recalk + precisionk != 0:
        F1k = 2*recalk*precisionk/(recalk + precisionk)
    else:
        F1k = 0
  
    resultados =  pd.DataFrame({
        'userId' : [userId],
        'precisionK': [precisionk],
        'recalK':[recalk],
        'F1K':[F1k]
        })
    return resultados
    
def resultadosUmbral(t, p=1):
    listaUs = train['userId'].unique()
    evaluacionSBERT = pd.DataFrame()
    for i in listaUs:
        evaluacionSBERT = pd.concat([evaluacionSBERT,evaluacion(i,t=t)], ignore_index=True)

    solucion = evaluacionSBERT[['precisionK', 'recalK','F1K']].mean()*p 
    return solucion

Resultado con t=3.5 y p=100, es decir, las métricas expresadas como porcentaje.

In [8]:
resultadosUmbral(3.5,100)

precisionK    0.180328
recalK        0.220887
F1K           0.162655
dtype: float64

## Predicción de rating
Se prueba un enfoque alternativo: en vez de recomendar las k películas más similares, se intenta predecir directamente el rating que un usuario daría a una película, mapeando la similitud del coseno (entre 0 y 1) a la escala de rating (entre 0.5 y 5).

Este enfoque se descarta, ya que no tiene mucho sentido: la relación entre similitud del coseno y rating no es necesariamente lineal, y el RMSE obtenido (en torno a 1.61) muestra que la predicción no es fiable.

In [9]:
def resultadosRating():
    listaUs = train['userId'].unique()
    resultados = pd.DataFrame()
    
    for i in listaUs:
        usuarioTest = test[test['userId'] == i]
        userVec = basedUserProfile(i)
        
        # Similitud con TODAS las películas de una vez
        simsAll = cosine_similarity(userVec.reshape(1, -1), embeddings).flatten()
        
        predicciones, reales = [], []
        for _, row in usuarioTest.iterrows():
            movieId = row['movieId']
            
            indPelicula = df[df['movieId'] == movieId].index[0]
            sim = simsAll[indPelicula]
            
            #  [0, 1] -> [0.5, 5]
            ratingPred = 0.5 + sim * 4.5
            
            predicciones.append(ratingPred)
            reales.append(row['rating'])
        
        if predicciones:
            reales = np.array(reales)
            predicciones = np.array(predicciones)
            rmse = np.sqrt(np.mean((reales - predicciones) ** 2))
            resultados = pd.concat([resultados, pd.DataFrame({
                'userId': [i],
                'RMSE': [rmse]
            })], ignore_index=True)
    
    print(resultados['RMSE'].mean())
    return resultados

resultados_rating = resultadosRating()

1.6146195147665254


## Evaluación alternativa: solo usuarios con intersección no vacía
En la evaluación anterior, muchos usuarios no tienen ninguna coincidencia entre sus recomendaciones y sus películas de test, por lo que su métrica se queda en 0 y penaliza la media global. Aquí se prueba una variante que solo tiene en cuenta a los usuarios cuya intersección entre recomendaciones y test no está vacía, para medir el rendimiento del modelo únicamente donde sí ha encontrado coincidencias.

In [18]:
#idea de evaluar en intersecciones no vacias:

def evaluacionMOD(userId, k=10, t=3.5):
    indRec = recomendacion(userId)
    movieIdRec = df.iloc[indRec]['movieId'].values
    interseccion = test[(test['userId']==userId) & (test['movieId'].isin(movieIdRec))]
    
    nRelevantes = (interseccion['rating'] >= t).sum()
    # nTest = (test[test["userId"]==userId]['rating'] >= t).sum() #pelis test relevantes
    if len(interseccion)==0:
        metrica = 0
        inter = 0
    else:
        metrica = nRelevantes/len(interseccion)
        inter = 1
  
    resultados =  pd.DataFrame({
        'userId' : [userId],
        'metrica': [metrica],
        'interseccon':[inter]
        })
    return resultados
    
def resultadosUmbralMOD(t):
    listaUs = train['userId'].unique()
    evaluacionSBERT = pd.DataFrame()
    for i in listaUs:
        evaluacionSBERT = pd.concat([evaluacionSBERT,evaluacionMOD(i,t=t)], ignore_index=True)
    
    solucion = evaluacionSBERT  
    return solucion

In [19]:
res = resultadosUmbralMOD(3.5)


In [12]:
res

,userId,metrica,interseccon
0,1,0.0,0
1,2,0.0,0
2,3,0.0,0
3,4,0.0,0
4,5,0.0,0
...,...,...,...
605,606,0.0,0
606,607,0.0,0
607,608,0.0,0
608,609,0.0,0


Se filtran los usuarios con intersección no vacía y se calcula la media de su métrica.

In [13]:
dfinterseccion = res[res["interseccon"]==1]
dfinterseccion['metrica'].mean()

np.float64(0.4782608695652174)

Resumen final: precisión media entre los usuarios con intersección, tasa de usuarios con intersección no vacía sobre el total, y cuántos usuarios entran en ese subconjunto.

In [20]:
tasa = res['interseccon'].mean()
n_validos = res['interseccon'].sum()
n_total = len(res)

print(f"PrecisionK: {dfinterseccion['metrica'].mean()*100:.4f}")
print(f"Tasa de intersección: {tasa:.4f}")
print(f"Usuarios válidos: {int(n_validos)} / {n_total}")

PrecisionK: 47.8261
Tasa de intersección: 0.0377
Usuarios válidos: 23 / 610
